# EDA with SQL

This notebook executes SQL queries to explore specific facts about the launch record dataset.

### Connect to the database

Installed in Environment:


pip install sqlalchemy==1.4.48  
pip install ipython-sql==0.4.1  
pip install ibm_db_sa  

**<span style="color:blueviolet;">First load the SQL extension and establish a connection with the database**

In [24]:
%load_ext sql

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [25]:
import csv, sqlite3

con = sqlite3.connect("my_data1.db")
cur = con.cursor()

**<span style="color:blueviolet;">Connect to magic sql using SQLite:**

In [26]:
%sql sqlite:///my_data1.db   

**<span style="color:blueviolet;">Write dataframe into a SQL database:**

In [27]:
import pandas as pd

# read in dataset
df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv")

In [28]:
df.shape

(101, 10)

In [29]:
df.to_sql("SPACEXTBL", con, if_exists='replace', index=False,method="multi")

101

**<span style="color:blueviolet;">Remove blank rows from table:**

In [30]:
%sql create table SPACEXTABLE as select * from SPACEXTBL where Date is not null

 * sqlite:///my_data1.db
(sqlite3.OperationalError) table SPACEXTABLE already exists
[SQL: create table SPACEXTABLE as select * from SPACEXTBL where Date is not null]
(Background on this error at: https://sqlalche.me/e/14/e3q8)


### First look of the dataset

In [8]:
#Retrieve and print all columns, without and with SQLiteMagic

statement = '''SELECT * FROM SPACEXTABLE limit 5'''
cur.execute(statement)
all = cur.fetchall()

for row in all:
    print(row)

('2010-06-04', '18:45:00', 'F9 v1.0  B0003', 'CCAFS LC-40', 'Dragon Spacecraft Qualification Unit', 0, 'LEO', 'SpaceX', 'Success', 'Failure (parachute)')
('2010-12-08', '15:43:00', 'F9 v1.0  B0004', 'CCAFS LC-40', 'Dragon demo flight C1, two CubeSats, barrel of Brouere cheese', 0, 'LEO (ISS)', 'NASA (COTS) NRO', 'Success', 'Failure (parachute)')
('2012-05-22', '7:44:00', 'F9 v1.0  B0005', 'CCAFS LC-40', 'Dragon demo flight C2', 525, 'LEO (ISS)', 'NASA (COTS)', 'Success', 'No attempt')
('2012-10-08', '0:35:00', 'F9 v1.0  B0006', 'CCAFS LC-40', 'SpaceX CRS-1', 500, 'LEO (ISS)', 'NASA (CRS)', 'Success', 'No attempt')
('2013-03-01', '15:10:00', 'F9 v1.0  B0007', 'CCAFS LC-40', 'SpaceX CRS-2', 677, 'LEO (ISS)', 'NASA (CRS)', 'Success', 'No attempt')


In [9]:
%sql SELECT * FROM SPACEXTABLE LIMIT 5;

 * sqlite:///my_data1.db
Done.


Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of Brouere cheese",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


In [10]:
## number of rows in the dataset
%sql SELECT COUNT(*) FROM SPACEXTABLE;

 * sqlite:///my_data1.db
Done.


COUNT(*)
101


### Explore Specific Features

Note: If the column names are in mixed case enclose it in double quotes
--- For Example "Landing_Outcome"

##### <span style="color:limegreen;">**Display the names of the unique launch sites  in the space mission**

In [11]:
%sql SELECT DISTINCT Launch_Site FROM SPACEXTABLE;

 * sqlite:///my_data1.db
Done.


Launch_Site
CCAFS LC-40
VAFB SLC-4E
KSC LC-39A
CCAFS SLC-40


##### <span style="color:limegreen;">**Display 5 records where launch sites begin with the string 'CCA'**

In [12]:
%sql SELECT * FROM SPACEXTABLE WHERE Launch_Site LIKE 'CCA%' LIMIT 5;

 * sqlite:///my_data1.db
Done.


Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of Brouere cheese",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


##### <span style="color:limegreen;">**Display the total payload mass carried by boosters launched by NASA (CRS)**

In [13]:
%sql SELECT SUM(PAYLOAD_MASS__KG_) FROM SPACEXTABLE WHERE Customer = 'NASA (CRS)';

 * sqlite:///my_data1.db
Done.


SUM(PAYLOAD_MASS__KG_)
45596


##### <span style="color:limegreen;">**Display average payload mass carried by booster version F9 v1.1**


In [14]:
%sql SELECT AVG(PAYLOAD_MASS__KG_) FROM SPACEXTABLE WHERE Booster_Version = 'F9 v1.1';

 * sqlite:///my_data1.db
Done.


AVG(PAYLOAD_MASS__KG_)
2928.4


##### <span style="color:limegreen;">**List the date when the first succesful landing outcome in ground pad was acheived.**

In [15]:
%sql SELECT MIN(Date) FROM SPACEXTABLE WHERE Landing_Outcome = 'Success (ground pad)';

 * sqlite:///my_data1.db
Done.


MIN(Date)
2015-12-22


##### <span style="color:limegreen;">**List the names of the boosters which have success in drone ship and have payload mass greater than 4000 but less than 6000**


In [16]:
%sql SELECT Booster_Version FROM SPACEXTABLE WHERE (PAYLOAD_MASS__KG_ BETWEEN 4000 AND 6000) \
AND Landing_Outcome = 'Success (drone ship)';

 * sqlite:///my_data1.db
Done.


Booster_Version
F9 FT B1022
F9 FT B1026
F9 FT B1021.2
F9 FT B1031.2


##### <span style="color:limegreen;">**List the total number of successful and failure mission outcomes**


In [17]:
%%sql SELECT Mission_Outcome, COUNT(Mission_Outcome) as total
FROM SPACEXTABLE
GROUP BY Mission_Outcome;

 * sqlite:///my_data1.db
Done.


Mission_Outcome,total
Failure (in flight),1
Success,98
Success,1
Success (payload status unclear),1


In [18]:
%%sql SELECT COUNT(*) FROM SPACEXTABLE
WHERE Mission_Outcome LIKE 'sUCCESS%';

 * sqlite:///my_data1.db
Done.


COUNT(*)
100


In [19]:
%%sql SELECT COUNT(*) FROM SPACEXTABLE
WHERE Mission_Outcome LIKE 'failure%';

 * sqlite:///my_data1.db
Done.


COUNT(*)
1


##### <span style="color:limegreen;">**List the names of the booster_versions which have carried the maximum payload mass.**


In [20]:
%%sql SELECT Booster_Version FROM SPACEXTABLE
WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTABLE);

 * sqlite:///my_data1.db
Done.


Booster_Version
F9 B5 B1048.4
F9 B5 B1049.4
F9 B5 B1051.3
F9 B5 B1056.4
F9 B5 B1048.5
F9 B5 B1051.4
F9 B5 B1049.5
F9 B5 B1060.2
F9 B5 B1058.3
F9 B5 B1051.6


In [21]:
%%sql SELECT Booster_Version, PAYLOAD_MASS__KG_ FROM SPACEXTABLE
WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTABLE);

 * sqlite:///my_data1.db
Done.


Booster_Version,PAYLOAD_MASS__KG_
F9 B5 B1048.4,15600
F9 B5 B1049.4,15600
F9 B5 B1051.3,15600
F9 B5 B1056.4,15600
F9 B5 B1048.5,15600
F9 B5 B1051.4,15600
F9 B5 B1049.5,15600
F9 B5 B1060.2,15600
F9 B5 B1058.3,15600
F9 B5 B1051.6,15600


##### <span style="color:limegreen;">**List the records which will display the month names, failure landing_outcomes in drone ship ,booster versions, launch_site for the months in year 2015.**

In [22]:
%%sql 
SELECT SUBSTR(Date,6,2) as Month, Landing_Outcome, Booster_Version, Launch_Site, Date
FROM SPACEXTABLE
WHERE SUBSTR(Date,1,4) = '2015' AND Landing_Outcome = 'Failure (drone ship)';

 * sqlite:///my_data1.db
Done.


Month,Landing_Outcome,Booster_Version,Launch_Site,Date
01,Failure (drone ship),F9 v1.1 B1012,CCAFS LC-40,2015-01-10
04,Failure (drone ship),F9 v1.1 B1015,CCAFS LC-40,2015-04-14


Note: SQLLite does not support monthnames. So need to use substr(Date, 4, 2) as month to get the months and substr(Date,7,4)='2015' for year.

##### <span style="color:limegreen;">**Rank the count of landing outcomes (such as Failure (drone ship) or Success (ground pad)) between the date 2010-06-04 and 2017-03-20, in descending order.**


In [23]:
%%sql
SELECT Landing_Outcome, COUNT(Landing_Outcome) as Count
FROM SPACEXTABLE
WHERE Date BETWEEN '2010-06-04' AND '2017-03-20'
GROUP BY Landing_Outcome
ORDER BY Count DESC;

 * sqlite:///my_data1.db
Done.


Landing_Outcome,Count
No attempt,10
Success (drone ship),5
Failure (drone ship),5
Success (ground pad),3
Controlled (ocean),3
Uncontrolled (ocean),2
Failure (parachute),2
Precluded (drone ship),1
